# 03a · Descărcare HR-VPP din WEkEO (rulează LOCAL, pe PC)
Productivitatea totală `TPROD`, sezonul 1 (`s1`), anii 2017–2024, interogare după `AOI_BBOX` → `data/raw/hrvpp/` (în afara git).

- Dataset: `EO:EEA:DAT:CLMS_HRVPP_VPP` (VPP, raster 10 m, în proiecția UTM a tile-ului Sentinel-2). Tile-ul T34TGP (EPSG:32634) acoperă singur toată zona, deși trece de 24° E.
- Credențiale din `.env`: `WEKEO_TOKEN` (din portalul WEkEO) sau `WEKEO_USER` și `WEKEO_PASSWORD`. Nu se afișează nicăieri.
- Termeni: `hda` îi acceptă automat la `search()`. Notebook-ul verifică întâi starea lor și se oprește dacă nu i-ai acceptat tu.
- Reluabil: produsele deja prezente în `data/raw/hrvpp/` nu se descarcă din nou.

In [1]:
# Doar local: găsește rădăcina proiectului și încarcă .env (valorile nu se afișează)
import os, sys, json, logging
assert 'google.colab' not in sys.modules, 'Rulează 03a pe kernelul local (PC), nu pe Colab.'
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
logging.basicConfig(level=logging.INFO)
logging.getLogger('hda').setLevel(logging.INFO)   # niciodată DEBUG: la DEBUG, hda scrie tokenul în log

DATASET_ID = 'EO:EEA:DAT:CLMS_HRVPP_VPP'          # verificat în catalogul STAC public WEkEO
PARAM_VAL, SEASON_VAL = 'TPROD', 's1'
YEARS = BASE_YEARS + [YEAR]
HRVPP_DIR = RAW / 'hrvpp'
HRVPP_DIR.mkdir(parents=True, exist_ok=True)
print('Lucrez în', os.getcwd(), '| ani:', YEARS)

Lucrez în C:\Users\lefpa\Downloads\gisday2026\gisday2026 | ani: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [2]:
# Client WEkEO + starea termenilor dataset-ului (fără a-i accepta)
import requests
from hda.api import SearchResults
from gisday_lib import wekeo_client
c = wekeo_client(os.environ)
try:
    terms = c.dataset(DATASET_ID)['terms']            # GET /datasets/{id} -> lista de termeni
except requests.HTTPError as e:
    raise RuntimeError(f'Autentificare/acces WEkEO eșuat (HTTP {e.response.status_code}). '
                       'Verifică WEKEO_TOKEN sau WEKEO_USER/WEKEO_PASSWORD în .env. Nu încerca ocoliri.') from None
neacceptati = []
for t in terms:
    meta = c.get('terms', t)                           # {term_id, title, href, abstract}
    try:
        ok = bool(c.get('termsaccepted', t).get('accepted'))
    except requests.HTTPError:                         # 404 = niciodată acceptat
        ok = False
    print('acceptat  ' if ok else 'NEACCEPTAT', '|', t, '|', meta.get('title'), '|', meta.get('href'))
    if not ok:
        neacceptati.append(t)

ACCEPT_TERMS = True    # Patrick a confirmat acceptarea termenilor CLMS pe 08.11.2026
if neacceptati and not ACCEPT_TERMS:
    raise RuntimeError(f'Oprire: termeni neacceptați {neacceptati}. Acceptă-i pe wekeo.copernicus.eu '
                       '(sau setează ACCEPT_TERMS = True) și rulează din nou celula.')
print('Termeni OK')

acceptat   | Copernicus_Land_Monitoring_Service_Data_Policy | Copernicus Land Monitoring Service Data Policy | https://wekeo2-prod-data-access-config.s3.waw3-2.cloudferro.com/terms/Copernicus%20Land%20Monitoring%20Service%20Data%20Policy.pdf
Termeni OK


In [3]:
# Parametrii reali ai dataset-ului (nu ghicim numele): căutăm câmpurile care admit 'TPROD' și 's1'
Q = c.metadata(DATASET_ID)                             # GET /dataaccess/queryable/{id}
props = Q.get('properties', {})

def valori(schema):
    """Valorile permise ale unui câmp (oneOf/const, oricât de imbricate)."""
    out = []
    if isinstance(schema, dict):
        for k, x in schema.items():
            out += [str(x)] if k == 'const' else valori(x)
    elif isinstance(schema, list):
        for x in schema:
            out += valori(x)
    return out

for name, sch in props.items():
    vals = valori(sch)
    print(f'{name:24s}', sch.get('type', ''), vals[:16] if vals else '')
PRODUCT_PARAM = next((n for n, s in props.items() if PARAM_VAL in valori(s)), None)
SEASON_PARAM = next((n for n, s in props.items() if SEASON_VAL in [v.lower() for v in valori(s)]), None)
assert PRODUCT_PARAM and SEASON_PARAM, 'Nu găsesc câmpul pentru produs/sezon în lista de mai sus — alege-l manual.'
SEASON_VALUE = next(v for v in valori(props[SEASON_PARAM]) if v.lower() == SEASON_VAL)
for k in ('bbox', 'startdate', 'enddate'):
    assert k in props, f'Câmpul {k!r} nu există în queryables (vezi lista de mai sus).'

def as_value(name, v):
    return [v] if props[name].get('type') == 'array' else v      # MultiSelect cere listă

print('produs:', PRODUCT_PARAM, '=', PARAM_VAL, '| sezon:', SEASON_PARAM, '=', SEASON_VALUE)

dataset_id               string ['EO:EEA:DAT:CLMS_HRVPP_VPP']
itemsPerPage             string 
startIndex               string 
uid                      string 
productType              string ['MINV', 'MAXD', 'LENGTH', 'SOSD', 'QFLAG', 'EOSV', 'TPROD', 'MAXV', 'AMPL', 'SOSV', 'LSLOPE', 'EOSD', 'RSLOPE', 'SPROD']
platformSerialIdentifier string ['S2A, S2B']
tileId                   string 
productVersion           string 
productGroupId           string ['s1', 's2']
resolution               string ['10']
processingDate           string 
bbox                     array 
startdate                string 
enddate                  string 
produs: productType = TPROD | sezon: productGroupId = s1


In [4]:
# Căutare pe ani; păstrăm doar produsele al căror startdate e în anul cerut
TILE = 'T34TGP'          # acoperă singur toată zona; T35TLJ ar dubla volumul pentru o fâșie deja acoperită
found = {}
for y in YEARS:
    q = {'dataset_id': DATASET_ID, 'bbox': list(AOI_BBOX),
         'startdate': f'{y}-01-01T00:00:00.000Z', 'enddate': f'{y}-12-31T23:59:59.999Z',
         PRODUCT_PARAM: as_value(PRODUCT_PARAM, PARAM_VAL), SEASON_PARAM: as_value(SEASON_PARAM, SEASON_VALUE)}
    res = c.search(q)
    found[y] = [r for r in res.results if str(r['properties'].get('startdate', ''))[:4] == str(y) and TILE in r['id']]
    print(y, f'{len(res.results)} rezultate, {len(found[y])} în an:', [r['id'] for r in found[y]])
summary = {y: [{'id': r['id'], 'startdate': r['properties'].get('startdate'),
                'enddate': r['properties'].get('enddate'), 'size': r['properties'].get('size')} for r in v]
           for y, v in found.items()}
(HRVPP_DIR / 'search_results.json').write_text(json.dumps(summary, indent=1), encoding='utf-8')

2017 2 rezultate, 1 în an: ['VPP_2017_S2_T34TGP-010m_V101_s1_TPROD']


2018 2 rezultate, 1 în an: ['VPP_2018_S2_T34TGP-010m_V101_s1_TPROD']


2019 2 rezultate, 1 în an: ['VPP_2019_S2_T34TGP-010m_V101_s1_TPROD']


2020 2 rezultate, 1 în an: ['VPP_2020_S2_T34TGP-010m_V101_s1_TPROD']


2021 2 rezultate, 1 în an: ['VPP_2021_S2_T34TGP-010m_V105_s1_TPROD']


2022 2 rezultate, 1 în an: ['VPP_2022_S2_T34TGP-010m_V105_s1_TPROD']


2023 2 rezultate, 1 în an: ['VPP_2023_S2_T34TGP-010m_V105_s1_TPROD']


2024 2 rezultate, 1 în an: ['VPP_2024_S2_T34TGP-010m_V105_s1_TPROD']


1434

In [5]:
# Descărcare (reluabilă): sare peste produsele complete de pe disc
import zipfile

from gisday_lib import is_complete

def present(r):
    return is_complete(HRVPP_DIR, r['id'], r['properties'].get('size'))     # o conexiune ruptă lasă un fișier mai mic

todo = [r for v in found.values() for r in v if not present(r)]
print(len(todo), 'de descărcat;', sum(map(len, found.values())) - len(todo), 'deja pe disc')
if todo:
    SearchResults(c, todo, DATASET_ID).download(str(HRVPP_DIR))   # erorile apar în log, nu ca excepții
for z in HRVPP_DIR.glob('*.zip'):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(HRVPP_DIR)
lipsa = [r['id'] for v in found.values() for r in v if not present(r)]
print('lipsă după descărcare:', lipsa or 'nimic')

1 de descărcat; 7 deja pe disc


C:\Users\lefpa\Downloads\gisday2026\gisday2026\.venv\Lib\site-packages\hda\api.py:731: UserWarning: Warning: 491 requests remaining out of 500.
  warnings.warn(msg)


INFO:hda.api:Download rate 7.6MB/s


INFO:hda.api:Successfully downloaded: None


lipsă după descărcare: nimic


In [6]:
# Verificare: fișierele descărcate și CRS-ul lor (tile-urile UTM 34 și 35 rămân în proiecția lor)
import rasterio, pandas as pd
rows = []
for p in sorted(HRVPP_DIR.glob('*.tif')):
    with rasterio.open(p) as s:
        rows.append((p.name, s.crs.to_string() if s.crs else None, s.width, s.height, s.dtypes[0], s.nodata,
                     s.scales[0], s.offsets[0], round(p.stat().st_size / 2**20, 1)))
T = pd.DataFrame(rows, columns=['fisier', 'crs', 'w', 'h', 'dtype', 'nodata', 'scale', 'offset', 'MB'])
print(T.to_string(index=False))
print(T['crs'].value_counts().to_string())

                                   fisier        crs     w     h  dtype  nodata  scale  offset    MB
VPP_2017_S2_T34TGP-010m_V101_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 191.9
VPP_2018_S2_T34TGP-010m_V101_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 190.9
VPP_2019_S2_T34TGP-010m_V101_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 186.2
VPP_2020_S2_T34TGP-010m_V101_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 187.0
VPP_2021_S2_T34TGP-010m_V105_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 191.5
VPP_2022_S2_T34TGP-010m_V105_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 187.7
VPP_2023_S2_T34TGP-010m_V105_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 190.2
VPP_2024_S2_T34TGP-010m_V105_s1_TPROD.tif EPSG:32634 10980 10980 uint16 65535.0    0.1     0.0 188.3
crs
EPSG:32634    8
